In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q nnunetv2==2.7.0
print("\n>>> RESTART RUNTIME, then run Cell 2. <<<")

NVIDIA A100-SXM4-80GB, 81920 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 17.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 137.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.6 M

In [1]:
import os, shutil, time, textwrap, py_compile
from google.colab import drive
drive.mount('/content/drive')

os.environ['nnUNet_compile']='F'
os.environ['nnUNet_raw']='/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw'
os.environ['nnUNet_preprocessed']='/mnt/local-scratch/nnUNet_preprocessed'   # unused by predict, set to avoid warnings
os.environ['nnUNet_results']='/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results'

# --- rewrite RCOversample trainer: predict -tr needs the class importable ---
import nnunetv2
target = os.path.join(os.path.dirname(nnunetv2.__file__),
                      'training','nnUNetTrainer','variants','nnUNetTrainerRCOversample.py')
code = textwrap.dedent('''\
    import torch
    from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
    from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper


    class nnUNetTrainerRCOversample(nnUNetTrainer):
        """ResEnc-L + foreground oversampling 0.90 + RC (label 4) CE upweight 3x."""

        def __init__(self, plans, configuration, fold, dataset_json,
                     device=torch.device('cuda')):
            super().__init__(plans, configuration, fold, dataset_json, device)
            self.num_epochs = 500
            self.oversample_foreground_percent = 0.90

        def _build_loss(self):
            base = super()._build_loss()
            tgt = base.loss if isinstance(base, DeepSupervisionWrapper) else base
            ce = getattr(tgt, 'ce', None)
            if ce is not None:
                n = self.label_manager.num_segmentation_heads
                w = torch.ones(n, device=self.device)
                if n > 4:
                    w[4] = 3.0
                ce.weight = w
            return base
''')
with open(target,'w') as f: f.write(code)
py_compile.compile(target, doraise=True)
print("RCOversample trainer written + compiles clean")

# --- stage val images to local disk (avoids the Drive-read stall) ---
SRC='/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'
LOCAL_IN='/content/imagesTs_local'
if not os.path.exists(LOCAL_IN):
    t0=time.time(); shutil.copytree(SRC, LOCAL_IN)
    print(f"staged {len(os.listdir(LOCAL_IN))} files in {(time.time()-t0)/60:.1f} min (want 716)")
else:
    print("already staged:", len(os.listdir(LOCAL_IN)))

# --- confirm all four models are present on Drive ---
R=os.environ['nnUNet_results']+'/Dataset501_BraTSMET2026'
for d,f in [('nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres',0),
            ('nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres',1),
            ('nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres',3),
            ('nnUNetTrainerRCOversample__nnUNetResEncUNetLPlans__3d_fullres',0)]:
    p=f'{R}/{d}/fold_{f}/checkpoint_best.pth'
    print(f"  fold_{f} {d.split('__')[0]}: {'OK' if os.path.exists(p) else 'MISSING'}")

os.makedirs('/mnt/local-scratch/preds', exist_ok=True)
!df -h /mnt/local-scratch | tail -1

Mounted at /content/drive
RCOversample trainer written + compiles clean


KeyboardInterrupt: 

In [2]:
import os, time
LOCAL_IN='/content/imagesTs_local'
SRC='/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'

n = len(os.listdir(LOCAL_IN)) if os.path.exists(LOCAL_IN) else 0
print(f"partial copy: {n}/716 files landed")

# nuke partial, redo with shell cp (much better over Drive FUSE)
!rm -rf "{LOCAL_IN}"
os.makedirs(LOCAL_IN, exist_ok=True)
t0=time.time()
!cp -r "{SRC}"/. "{LOCAL_IN}"/
print(f"staged {len(os.listdir(LOCAL_IN))} files in {(time.time()-t0)/60:.1f} min (want 716)")

partial copy: 8/716 files landed
^C
staged 259 files in 0.4 min (want 716)


In [3]:
import os, time
LOCAL_IN='/content/imagesTs_local'
SRC='/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_raw/Dataset501_BraTSMET2026/imagesTs'
!rm -rf "{LOCAL_IN}"; mkdir -p "{LOCAL_IN}"
t0=time.time()
!cp -r "{SRC}"/. "{LOCAL_IN}"/
print(f"staged {len(os.listdir(LOCAL_IN))} files in {(time.time()-t0)/60:.1f} min (want 716)")

staged 716 files in 0.9 min (want 716)


In [4]:
R='/content/drive/MyDrive/BraTS/BraTS2026/nnUNet_results/Dataset501_BraTSMET2026'
for tr,f in [('nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres',0),
             ('nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres',1),
             ('nnUNetTrainer_500epochs__nnUNetResEncUNetLPlans__3d_fullres',3),
             ('nnUNetTrainerRCOversample__nnUNetResEncUNetLPlans__3d_fullres',0)]:
    p=f'{R}/{tr}/fold_{f}/checkpoint_best.pth'
    print(f"  {tr.split('__')[0]} fold_{f}: {'OK' if os.path.exists(p) else 'MISSING'}")

  nnUNetTrainer_500epochs fold_0: OK
  nnUNetTrainer_500epochs fold_1: OK
  nnUNetTrainer_500epochs fold_3: OK
  nnUNetTrainerRCOversample fold_0: OK


In [5]:
import os, time
os.environ['nnUNet_compile']='F'
OUT='/mnt/local-scratch/preds/f0'; os.makedirs(OUT, exist_ok=True)
t0=time.time()
!nnUNetv2_predict -i /content/imagesTs_local -o "{OUT}" \
  -d 501 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_500epochs \
  -f 0 -chk checkpoint_best.pth --save_probabilities -npp 2 -nps 2
print(f"\nfold 0 done in {(time.time()-t0)/60:.1f} min | files: {len(os.listdir(OUT))}")


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict

Predicting BraTS-MET-00833-000:
perform_everything_on_device: True
100% 1/1 [00:17<00:00, 17.51s/it]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00833-000

Predicting BraTS-MET-00834-000:
perform_everything_on_device: True
100% 1/1 [00:00<00:00,  1.61it/s]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00834-000

Predicting BraTS-MET-00835

In [6]:
import os, time
os.environ['nnUNet_compile']='F'
jobs = [('f0','nnUNetTrainer_500epochs',0), ('f1','nnUNetTrainer_500epochs',1),
        ('f3','nnUNetTrainer_500epochs',3), ('rc','nnUNetTrainerRCOversample',0)]
for name, tr, f in jobs:
    OUT=f'/mnt/local-scratch/preds/{name}'; os.makedirs(OUT, exist_ok=True)
    if len(os.listdir(OUT)) >= 358:
        print(f"{name}: already done, skipping"); continue
    t0=time.time()
    !nnUNetv2_predict -i /content/imagesTs_local -o "{OUT}" \
      -d 501 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr {tr} \
      -f {f} -chk checkpoint_best.pth --save_probabilities -npp 2 -nps 2
    print(f"\n=== {name} done in {(time.time()-t0)/60:.1f} min | files: {len(os.listdir(OUT))} ===")
print("ALL PREDICTIONS DONE")


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict

Predicting BraTS-MET-00833-000:
perform_everything_on_device: True
100% 1/1 [00:16<00:00, 16.90s/it]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00833-000

Predicting BraTS-MET-00834-000:
perform_everything_on_device: True
100% 1/1 [00:00<00:00,  1.61it/s]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00834-000

Predicting BraTS-MET-00835

In [8]:
import os, time, zipfile
os.environ['nnUNet_compile']='F'
DRIVE='/content/drive/MyDrive/BraTS/BraTS2026/predictions'
os.makedirs(DRIVE, exist_ok=True)

# --- 1. predictions (skip-safe) ---
jobs = [('f0','nnUNetTrainer_500epochs',0), ('f1','nnUNetTrainer_500epochs',1),
        ('f3','nnUNetTrainer_500epochs',3), ('rc','nnUNetTrainerRCOversample',0)]
for name, tr, f in jobs:
    OUT=f'/mnt/local-scratch/preds/{name}'; os.makedirs(OUT, exist_ok=True)
    if len(os.listdir(OUT)) >= 358:
        print(f"{name}: already done, skipping"); continue
    t0=time.time()
    !nnUNetv2_predict -i /content/imagesTs_local -o "{OUT}" \
      -d 501 -c 3d_fullres -p nnUNetResEncUNetLPlans -tr {tr} \
      -f {f} -chk checkpoint_best.pth --save_probabilities -npp 2 -nps 2
    print(f"=== {name}: {(time.time()-t0)/60:.1f} min, {len(os.listdir(OUT))} files ===")

# --- 2. ensembles ---
P='/mnt/local-scratch/preds'
combos = {
    'ens_rc_f0f1f3': [f'{P}/rc', f'{P}/f0', f'{P}/f1', f'{P}/f3'],
    'ens_f0f1f3'   : [f'{P}/f0', f'{P}/f1', f'{P}/f3'],
    'ens_rc_f3'    : [f'{P}/rc', f'{P}/f3'],
}
for name, ins in combos.items():
    OUT=f'/mnt/local-scratch/{name}'; os.makedirs(OUT, exist_ok=True)
    t0=time.time()
    !nnUNetv2_ensemble -i {" ".join(ins)} -o "{OUT}" -np 4
    print(f"=== {name}: {(time.time()-t0)/60:.1f} min, {len(os.listdir(OUT))} files ===")

# --- 3. zip + persist to Drive (small, survives a drop) ---
for name in combos:
    SRC=f'/mnt/local-scratch/{name}'
    niis=sorted(f for f in os.listdir(SRC) if f.endswith('.nii.gz'))
    Z=f'{DRIVE}/BraTS_Uniandes_{name}.zip'
    with zipfile.ZipFile(Z,'w',zipfile.ZIP_DEFLATED) as z:
        for f in niis: z.write(f'{SRC}/{f}', arcname=f)
    print(f"{name}: zipped {len(niis)} -> Drive")
print("\nALL DONE — zips on Drive, safe from disconnect")


#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 179 cases in the source folder
I am processing 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 179 cases that I would like to predict

Predicting BraTS-MET-00833-000:
perform_everything_on_device: True
100% 1/1 [00:16<00:00, 16.50s/it]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00833-000

Predicting BraTS-MET-00834-000:
perform_everything_on_device: True
100% 1/1 [00:00<00:00,  1.61it/s]
sending off prediction to background worker for resampling and export
done with BraTS-MET-00834-000

Predicting BraTS-MET-00835

In [9]:
import os, numpy as np, nibabel as nib
for name in ['ens_rc_f0f1f3','ens_f0f1f3','ens_rc_f3']:
    P=f'/mnt/local-scratch/{name}'
    niis=sorted(f for f in os.listdir(P) if f.endswith('.nii.gz'))
    counts={}; empty=0
    for f in niis:
        labs=np.unique(np.asanyarray(nib.load(f'{P}/{f}').dataobj)).astype(int).tolist()
        if labs==[0]: empty+=1
        for l in labs: counts[l]=counts.get(l,0)+1
    print(f"{name}: files={len(niis)} empty={empty} presence={dict(sorted(counts.items()))}")
print("  reference: 0+1+3 originally had RC in 65/179 -> LB 0.394")

ens_rc_f0f1f3: files=179 empty=14 presence={0: 179, 1: 82, 2: 152, 3: 163, 4: 61}
ens_f0f1f3: files=179 empty=15 presence={0: 179, 1: 82, 2: 152, 3: 163, 4: 65}
ens_rc_f3: files=179 empty=15 presence={0: 179, 1: 81, 2: 154, 3: 162, 4: 66}
  reference: 0+1+3 originally had RC in 65/179 -> LB 0.394


In [10]:
import os
from google.colab import files

DRIVE='/content/drive/MyDrive/BraTS/BraTS2026/predictions'
for name in ['ens_rc_f0f1f3', 'ens_rc_f3']:
    Z=f'{DRIVE}/BraTS_Uniandes_{name}.zip'
    print(f"{name}: exists={os.path.exists(Z)} size={os.path.getsize(Z)/1e6:.0f} MB")

for name in ['ens_rc_f0f1f3', 'ens_rc_f3']:
    files.download(f'{DRIVE}/BraTS_Uniandes_{name}.zip')
print("downloading both...")

ens_rc_f0f1f3: exists=True size=1 MB
ens_rc_f3: exists=True size=2 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloading both...
